In [ ]:
# 這是使用dict來定義每個象限的資訊，讓程式碼更有結構且易於維護。
# 每個象限的資訊包含標題、市場特徵、代表標的、策略建議，以及必要的警告提示。
class MarketQuadrantAnalyzer:
    def __init__(self):
        # 定義各象限的輸出文案
        self.quadrant_info = {
            1: {
                "title": "第一象限 (右上)：擴張 (+) + 高波動 (+) ——「狂熱末升段」",
                "features": "市場特徵： 景氣很好，但價格已經不便宜，多空分歧大，洗盤頻繁。",
                "targets": "代表標的： 低軌衛星概念股（如耀華）在題材最熱、成交量暴增的時候。",
                "strategy": "策略： 動能策略 (Momentum)。只要趨勢沒斷就追，但必須嚴格執行移動止損。",
                "warning": "！⚠️在第一象限 (狂熱末升段)，雖然同樣是擴張，但高波動代表風險溢酬正在縮減，此時模型應自動調降倉位並縮緊移動止損。"
            },
            2: {
                "title": "第二象限 (左上)：收縮 (-) + 高波動 (+) ——「危機恐慌期」",
                "features": "市場特徵： 基本面轉壞（如景氣燈號轉藍），伴隨市場恐慌拋售。",
                "targets": "代表標的： 遭遇黑天鵝事件時的大盤，或是產業景氣崩跌時。",
                "strategy": "策略： 均值回歸 (Mean Reversion) 或放空。利用超跌後的反彈（乖離率過大）獲利。"
            },
            3: {
                "title": "第三象限 (左下)：收縮 (-) + 低波動 (-) ——「蕭條築底期」",
                "features": "市場特徵： 市場極度冷清，乏人問津。這就是你研究日本經濟泡沫後那長達十幾年的狀態。",
                "targets": "代表標的： 傳統產業、高股息、或是在谷底盤整多年的週期股。",
                "strategy": "策略： 價值投資。尋找股價淨值比 P/B<1 且現金流穩定的公司。"
            },
            4: {
                "title": "第四象限 (右下)：擴張 (+) + 低波動 (-) ——「金髮女孩 (Goldilocks)」",
                "features": "市場特徵： 經濟穩步復甦，通膨適中，是投資者最舒服的「送分題」區間。",
                "targets": "代表標的： 進入穩定成長期的權值股、獲利開始翻正的科技股。",
                "strategy": "策略： 趨勢追蹤 (Trend Following)。這是你的量化模型最應該優先捕捉的區間。",
                "warning": "！⚠️可以調高凱利公式的投入比例，並放寬止損區間。"
            }
        }

    def calculate_x_score(self, price, ma200, adx, yoy_t2, yoy_t1, yoy_now, rsi):
        """計算 X 軸 (趨勢/經濟動能) 分數"""
        x_score = 0
        
        # 1. Price vs MA200
        if price > ma200:
            x_score += 2
        elif price < ma200:
            x_score -= 2
            
        # 2. ADX (14)
        if adx > 25:
            x_score += 1
        elif adx < 20:
            x_score -= 1
            
        # 3. 營收 YoY 連續兩季增長
        if yoy_t2 < yoy_t1 < yoy_now:
            x_score += 2
        else:
            x_score -= 2
            
        # 4. RSI
        if rsi > 70:
            x_score += 2
        elif rsi > 50:
            x_score += 1
        elif rsi < 50:
            x_score -= 1
            
        return x_score

    def calculate_y_score(self, atr, atr_60d_avg, bbw_percentile, vix_percentile, hv_percentile):
        """計算 Y 軸 (波動) 分數"""
        y_score = 0
        
        # 1. ATR %
        if atr > atr_60d_avg:
            y_score += 2
        else:
            y_score -= 2
            
        # 2. Bollinger Band Width
        if bbw_percentile > 0.8:
            y_score += 2
        elif bbw_percentile < 0.2:
            y_score -= 2
            
        # 3. VIX (隱含波動率) 位於前 25% (即分位數 >= 0.75)
        if vix_percentile >= 0.75:
            y_score += 1
        else:
            y_score -= 1
            
        # 4. 歷史波動率 (HV) 位於前 25% (即分位數 >= 0.75)
        if hv_percentile >= 0.75:
            y_score += 1
        else:
            y_score -= 1
            
        return y_score

    def print_quadrant_info(self, quadrant):
        """輸出象限對應的策略資訊"""
        info = self.quadrant_info[quadrant]
        print(info["title"])
        print(info["features"])
        print(info["targets"])
        print(info["strategy"])
        if "warning" in info:
            print(info["warning"])
        print("-" * 50)

    def analyze(self, data_point):
        """執行整體評估並輸出結果"""
        # 取得 X 與 Y 座標總分
        x = self.calculate_x_score(
            price=data_point['price'], 
            ma200=data_point['ma200'], 
            adx=data_point['adx'], 
            rsi=data_point['rsi'],
            yoy_t2=data_point['yoy_t2'],
            yoy_t1=data_point['yoy_t1'],
            yoy_now=data_point['yoy_now']
        )
        
        y = self.calculate_y_score(
            atr=data_point['atr'], 
            atr_60d_avg=data_point['atr_60d_avg'], 
            bbw_percentile=data_point['bbw_percentile'], 
            vix_percentile=data_point['vix_percentile'], 
            hv_percentile=data_point['hv_percentile']
        )
        
        print(f"指標計分結果 -> X (動能): {x}, Y (波動): {y}")
        
        # 判斷象限
        if x > 0 and y > 0:
            self.print_quadrant_info(1)
        elif x < 0 and y > 0:
            self.print_quadrant_info(2)
        elif x < 0 and y < 0:
            self.print_quadrant_info(3)
        elif x > 0 and y < 0:
            self.print_quadrant_info(4)
        else:
            # X 或 Y 其中一者為 0，落在座標軸或原點
            print("位於原點或座標軸上(0)，代表象限不明，不予操作。")
            print("-" * 50)

# 實例化模組
analyzer = MarketQuadrantAnalyzer()
# 測試標的 A：符合第一象限 (狂熱末升段)，動能極強，且波動率處於極端擴張狀態
sample_data_A = {
    'price': 150, 'ma200': 100,          # X: +2 (價格站上均線)
    'adx': 30,                           # X: +1 (強趨勢)
    'yoy_t2': 5, 'yoy_t1': 10, 'yoy_now': 15, # X: +2 (營收連續遞增)
    'rsi': 75,                           # X: +2 (RSI > 70)
    # => X 總分 = 7

    'atr': 5.0, 'atr_60d_avg': 3.0,      # Y: +2 (波動大於均值)
    'bbw_percentile': 0.9,               # Y: +2 (布林通道極度擴張)
    'vix_percentile': 0.8,               # Y: +1 (隱含波動率在前 25%)
    'hv_percentile': 0.8                 # Y: +1 (歷史波動率在前 25%)
    # => Y 總分 = 6
}
print("【測試標的 A】")
analyzer.analyze(sample_data_A)

# 測試標的 B：符合第四象限 (金髮女孩)，穩步向上的趨勢，且波動率極低
sample_data_B = {
    'price': 120, 'ma200': 100,          # X: +2 (價格站上均線)
    'adx': 28,                           # X: +1 (強趨勢)
    'yoy_t2': 2, 'yoy_t1': 5, 'yoy_now': 8,   # X: +2 (營收連續遞增)
    'rsi': 60,                           # X: +1 (RSI > 50)
    # => X 總分 = 6

    'atr': 1.5, 'atr_60d_avg': 2.0,      # Y: -2 (波動小於均值)
    'bbw_percentile': 0.1,               # Y: -2 (布林通道擠壓)
    'vix_percentile': 0.3,               # Y: -1 (未達前 25%)
    'hv_percentile': 0.4                 # Y: -1 (未達前 25%)
    # => Y 總分 = -6
}
print("【測試標的 B】")
analyzer.analyze(sample_data_B)

# 測試標的 C：落在原點 (0, 0)，多空動能抵銷，且波動率指標互相矛盾
sample_data_C = {
    'price': 90, 'ma200': 100,           # X: -2 (跌破均線)
    'adx': 22,                           # X:  0 (落在 20~25 之間)
    'yoy_t2': 5, 'yoy_t1': 10, 'yoy_now': 15, # X: +2 (營收基本面卻很好)
    'rsi': 50,                           # X:  0 (中性)
    # => X 總分 = 0

    'atr': 3.0, 'atr_60d_avg': 2.0,      # Y: +2 (近期 ATR 升高)
    'bbw_percentile': 0.5,               # Y:  0 (布林通道寬度適中)
    'vix_percentile': 0.4,               # Y: -1 (VIX 未達前 25%)
    'hv_percentile': 0.3                 # Y: -1 (HV 未達前 25%)
    # => Y 總分 = 0
}
print("【測試標的 C】")
analyzer.analyze(sample_data_C)

In [ ]:
import pandas as pd
import numpy as np

class MarketQuadrantAnalyzer:
    def __init__(self):
        # 定義各象限的輸出文案
        self.quadrant_info = {
            1: {
                "title": "第一象限 (右上)：擴張 (+) + 高波動 (+) ——「狂熱末升段」",
                "features": "市場特徵： 景氣很好，但價格已經不便宜，多空分歧大，洗盤頻繁。",
                "targets": "代表標的： 低軌衛星概念股（如耀華）在題材最熱、成交量暴增的時候。",
                "strategy": "策略： 動能策略 (Momentum)。只要趨勢沒斷就追，但必須嚴格執行移動止損。",
                "warning": "！⚠️在第一象限 (狂熱末升段)，雖然同樣是擴張，但高波動代表風險溢酬正在縮減，此時模型應自動調降倉位並縮緊移動止損。"
            },
            2: {
                "title": "第二象限 (左上)：收縮 (-) + 高波動 (+) ——「危機恐慌期」",
                "features": "市場特徵： 基本面轉壞（如景氣燈號轉藍），伴隨市場恐慌拋售。",
                "targets": "代表標的： 遭遇黑天鵝事件時的大盤，或是產業景氣崩跌時。",
                "strategy": "策略： 均值回歸 (Mean Reversion) 或放空。利用超跌後的反彈（乖離率過大）獲利。"
            },
            3: {
                "title": "第三象限 (左下)：收縮 (-) + 低波動 (-) ——「蕭條築底期」",
                "features": "市場特徵： 市場極度冷清，乏人問津。這就是你研究日本經濟泡沫後那長達十幾年的狀態。",
                "targets": "代表標的： 傳統產業、高股息、或是在谷底盤整多年的週期股。",
                "strategy": "策略： 價值投資。尋找股價淨值比 P/B<1 且現金流穩定的公司。"
            },
            4: {
                "title": "第四象限 (右下)：擴張 (+) + 低波動 (-) ——「金髮女孩 (Goldilocks)」",
                "features": "市場特徵： 經濟穩步復甦，通膨適中，是投資者最舒服的「送分題」區間。",
                "targets": "代表標的： 進入穩定成長期的權值股、獲利開始翻正的科技股。",
                "strategy": "策略： 趨勢追蹤 (Trend Following)。這是你的量化模型最應該優先捕捉的區間。",
                "warning": "！⚠️可以調高凱利公式的投入比例，並放寬止損區間。"
            }
        }
        pass

    def analyze_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        接收 Pandas DataFrame 並使用向量化運算，
        一次性計算所有時間點的 X、Y 分數與所屬象限。
        """
        # 複製一份資料避免改動到原始 DataFrame
        result_df = df.copy()
        
        # ---------------------------------------------------------
        # 計算 X 軸 (趨勢/經濟動能) 分數
        # ---------------------------------------------------------
        x_score = pd.Series(0, index=df.index)
        
        # 1. Price vs MA200
        cond_price = [df['price'] > df['ma200'], df['price'] < df['ma200']]
        x_score += np.select(cond_price, [2, -2], default=0)
        
        # 2. ADX (14)
        cond_adx = [df['adx'] > 25, df['adx'] < 20]
        x_score += np.select(cond_adx, [1, -1], default=0)
        
        # 3. 營收 YoY 遞增 (yoy_t2 < yoy_t1 < yoy_now)
        cond_yoy = (df['yoy_t2'] < df['yoy_t1']) & (df['yoy_t1'] < df['yoy_now'])
        x_score += np.where(cond_yoy, 2, -2)
        
        # 4. RSI
        cond_rsi = [df['rsi'] > 70, df['rsi'] > 50, df['rsi'] < 50]
        x_score += np.select(cond_rsi, [2, 1, -1], default=0)
        
        result_df['X_Score'] = x_score

        # ---------------------------------------------------------
        # 計算 Y 軸 (波動) 分數
        # ---------------------------------------------------------
        y_score = pd.Series(0, index=df.index)
        
        # 1. ATR %
        y_score += np.where(df['atr'] > df['atr_60d_avg'], 2, -2)
        
        # 2. Bollinger Band Width Percentile
        cond_bbw = [df['bbw_percentile'] > 0.8, df['bbw_percentile'] < 0.2]
        y_score += np.select(cond_bbw, [2, -2], default=0)
        
        # 3. VIX Percentile
        y_score += np.where(df['vix_percentile'] >= 0.75, 1, -1)
        
        # 4. HV Percentile
        y_score += np.where(df['hv_percentile'] >= 0.75, 1, -1)
        
        result_df['Y_Score'] = y_score

        # ---------------------------------------------------------
        # 判斷象限
        # ---------------------------------------------------------
        cond_quadrant = [
            (result_df['X_Score'] > 0) & (result_df['Y_Score'] > 0), # 第一象限
            (result_df['X_Score'] < 0) & (result_df['Y_Score'] > 0), # 第二象限
            (result_df['X_Score'] < 0) & (result_df['Y_Score'] < 0), # 第三象限
            (result_df['X_Score'] > 0) & (result_df['Y_Score'] < 0)  # 第四象限
        ]
        # 標記象限，0 代表位於原點或座標軸上
        result_df['Quadrant'] = np.select(cond_quadrant, [1, 2, 3, 4], default=0)

        return result_df

In [ ]:
from quadrant import Quadrant
# 建立一份包含多天歷史資料的模擬 DataFrame
data = {
    'price': [150, 120, 90],
    'ma200': [100, 100, 100],
    'adx': [30, 28, 22],
    'yoy_t2': [5, 2, 5],
    'yoy_t1': [10, 5, 10],
    'yoy_now': [15, 8, 15],
    'rsi': [75, 60, 50],
    'atr': [5.0, 1.5, 3.0],
    'atr_60d_avg': [3.0, 2.0, 2.0],
    'bbw_percentile': [0.9, 0.1, 0.5],
    'vix_percentile': [0.8, 0.3, 0.4],
    'hv_percentile': [0.8, 0.4, 0.3]
}
df_history = pd.DataFrame(data, index=['2024-01-01', '2024-01-02', '2024-01-03'])

# 實例化模組並傳入 DataFrame
analyzer = Quadrant.MarketQuadrantAnalyzer()
df_result = analyzer.analyze_dataframe(df_history)

# 顯示計算結果的特定欄位
print(df_result[['X_Score', 'Y_Score', 'Quadrant']])

In [ ]:
#更簡潔的結果輸出版本
def get_quadrant_info(q):
    """一次性取得所有象限資訊並回傳為 Pandas Series"""
    if q in analyzer.quadrant_info:
        info = analyzer.quadrant_info[q]
        return pd.Series({
            'quadrant_title': info.get('title', ''),
            'quadrant_features': info.get('features', ''),
            'quadrant_targets': info.get('targets', ''),
            'quadrant_strategy': info.get('strategy', '')
        })
    else:
        # 若 q 為 0 或找不到，填入預設文字
        default_msg = "位於原點或座標軸上(0)，代表象限不明，不予操作。"
        return pd.Series({
            'quadrant_title': default_msg,
            'quadrant_features': default_msg,
            'quadrant_targets': default_msg,
            'quadrant_strategy': default_msg
        })

# 透過一次 .apply() 直接擴充四個新欄位
new_columns = ['quadrant_title', 'quadrant_features', 'quadrant_targets', 'quadrant_strategy']
df_result[new_columns] = df_result['Quadrant'].apply(get_quadrant_info)

# 橫向印出最後五天狀態
display(df_result[['Quadrant'] + new_columns].tail())

In [ ]:
#更簡潔的結果輸出版本之二
# 定義原點 (0) 的預設文字
default_text = "位於原點或座標軸上(0)，代表象限不明，不予操作。"

# 1. 建立欄位 (使用 map 快速映射)
# 透過 analyzer.quadrant_info 字典直接對應，如果找不到 (如 0) 則填入 default_text
df_result['quadrant_title'] = df_result['Quadrant'].map(
    lambda x: analyzer.quadrant_info.get(x, {}).get('title', default_text)
)
df_result['quadrant_features'] = df_result['Quadrant'].map(
    lambda x: analyzer.quadrant_info.get(x, {}).get('features', default_text)
)
df_result['quadrant_targets'] = df_result['Quadrant'].map(
    lambda x: analyzer.quadrant_info.get(x, {}).get('targets', default_text)
)
df_result['quadrant_strategy'] = df_result['Quadrant'].map(
    lambda x: analyzer.quadrant_info.get(x, {}).get('strategy', default_text)
)

# 2. 橫向列印設定
pd.set_option('display.max_colwidth', 50) # 限制每欄最多顯示 50 字，避免表格太寬爆版
# 若您希望文字完全不換行，請將上面 50 改為 None

# 3. 印出結果
cols = ['Quadrant', 'quadrant_title', 'quadrant_features', 'quadrant_targets', 'quadrant_strategy']
display(df_result[cols].tail())

In [ ]:
# 查看「最後一天」的詳細策略 (將橫向轉為直向)
last_day = df_result.iloc[-1] # 取最後一筆
print(last_day[['quadrant_title', 'quadrant_features', 'quadrant_strategy']].to_frame())

In [ ]:
#自己算指標
import pandas as pd
import numpy as np
import pandas_ta as ta

def prepare_quadrant_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    輸入包含基本 K 線 (Open, High, Low, Close) 的 DataFrame，
    計算象限模組所需的所有技術與波動率指標。
    """
    df = df.copy()

    # ---------------------------------------------------------
    # 1. 基礎技術指標 (X軸動能相關)
    # ---------------------------------------------------------
    df['price'] = df['Close']
    df['ma200'] = ta.sma(df['Close'], length=200)
    
    # ADX 計算會回傳多個欄位，我們只需要 ADX 本身
    adx_df = ta.adx(df['High'], df['Low'], df['Close'], length=14)
    df['adx'] = adx_df['ADX_14'] if adx_df is not None else np.nan
    
    df['rsi'] = ta.rsi(df['Close'], length=14)

    # 營收 YoY 模擬處理 (實務上這需要與基本面月營收資料庫 Merge)
    # 此處假設您的 DataFrame 已經有 'revenue_yoy' 欄位，且為月頻率或季頻率
    # df['yoy_t2'] = df['revenue_yoy'].shift(2)
    # df['yoy_t1'] = df['revenue_yoy'].shift(1)
    # df['yoy_now'] = df['revenue_yoy']

    # ---------------------------------------------------------
    # 2. 波動率指標 (Y軸波動相關)
    # ---------------------------------------------------------
    df['atr'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    df['atr_60d_avg'] = df['atr'].rolling(window=60).mean()

    # 布林通道寬度 (Bandwidth)
    bbands = ta.bbands(df['Close'], length=20, std=2)
    df['bbw'] = bbands['BBB_20_2.0'] if bbands is not None else np.nan

    # 歷史波動率 (Historical Volatility, HV) - 使用 20 日對數報酬率標準差年化
    log_returns = np.log(df['Close'] / df['Close'].shift(1))
    df['hv'] = log_returns.rolling(window=20).std() * np.sqrt(252)

    # ---------------------------------------------------------
    # 3. 滾動分位數 (Rolling Percentile) 計算
    # ---------------------------------------------------------
    # 使用過去 252 個交易日 (約一年) 作為滾動視窗計算當前數值在過去一年中的百分位
    # 邏輯：(過去視窗內 <= 當前值的數量) / 視窗總數量
    window_252 = 252
    
    def calc_rolling_percentile(series):
        return series.rolling(window=window_252).apply(
            lambda x: (x <= x.iloc[-1]).mean(), raw=False
        )

    df['bbw_percentile'] = calc_rolling_percentile(df['bbw'])
    df['hv_percentile'] = calc_rolling_percentile(df['hv'])
    
    # 註：VIX 為外部大盤指數，需另外抓取並 Merge 進 df 後再計算分位數
    # 假設已 Merge 進來為 'vix' 欄位
    # df['vix_percentile'] = calc_rolling_percentile(df['vix'])

    return df

In [ ]:
# 主程式流程，使用模組整合流程
# 1. 載入套件與自訂模組
import pandas as pd
from quadrant import Quadrant

# 2. 顯示設定 (避免文字截斷)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# 3. 準備您的歷史資料 df_history (此處省略抓資料的過程)
# df_history = ... 

# 4. 執行模型
analyzer = Quadrant.MarketQuadrantAnalyzer()
df_analyzed = analyzer.analyze_dataframe(df_history)

# 5. 附加文字說明 (呼叫剛寫好的擴充方法)
df_final = analyzer.attach_descriptions(df_analyzed)

# 6. 檢視結果
cols_to_show = ['Quadrant', 'quadrant_title', 'quadrant_strategy', 'quadrant_features', 'quadrant_targets']
display(df_final[cols_to_show].tail()) # Jupyter 中建議用 display 代替 print

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
print("目前筆記本所在目錄：", os.getcwd())
print("資料夾內的檔案：", os.listdir())

In [ ]:
#2026.3.17修改前
import pandas as pd
import numpy as np

class MarketQuadrantAnalyzer:
    def __init__(self):
        # 定義各象限的輸出文案
        self.quadrant_info = {
            1: {
                "title": "第一象限 (右上)：擴張 (+) + 高波動 (+) ——「狂熱末升段」",
                "features": "市場特徵： 景氣很好，價格不便宜，多空分歧大，洗盤頻繁。",
                "targets": "代表標的： 低軌衛星等概念股在題材最熱、成交量暴增時。",
                "strategy": "策略： 動能策略。只要趨勢沒斷就追，嚴格執行移動止損。"
            },
            2: {
                "title": "第二象限 (左上)：收縮 (-) + 高波動 (+) ——「危機恐慌期」",
                "features": "市場特徵： 基本面轉壞，伴隨市場恐慌拋售。",
                "targets": "代表標的： 遭遇黑天鵝事件時的大盤，或是產業崩跌時。",
                "strategy": "策略： 均值回歸或放空。利用超跌後的反彈獲利。"
            },
            3: {
                "title": "第三象限 (左下)：收縮 (-) + 低波動 (-) ——「蕭條築底期」",
                "features": "市場特徵： 市場極度冷清，乏人問津。",
                "targets": "代表標的： 傳統產業、高股息、或谷底盤整的週期股。",
                "strategy": "策略： 價值投資。尋找 P/B<1 且現金流穩定的公司。"
            },
            4: {
                "title": "第四象限 (右下)：擴張 (+) + 低波動 (-) ——「金髮女孩」",
                "features": "市場特徵： 經濟穩步復甦，通膨適中，最舒服的送分題區間。",
                "targets": "代表標的： 進入穩定成長期的權值股、獲利翻正的科技股。",
                "strategy": "策略： 趨勢追蹤。量化模型最應優先捕捉的區間。"
            }
        }

    def analyze_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """執行向量化運算，計算 X, Y 分數與所屬象限"""
        result_df = df.copy()
        
        # --- 計算 X 軸 ---
        x_score = pd.Series(0, index=df.index)
        x_score += np.select([df['adj_price'] > df['ma200'], df['adj_price'] < df['ma200']], [2, -2], default=0)
        x_score += np.select([df['adx'] > 25, df['adx'] < 20], [1, -1], default=0)
        
        cond_yoy = (df['yoy_t2'] < df['yoy_t1']) & (df['yoy_t1'] < df['yoy_now'])
        x_score += np.where(cond_yoy, 2, -2)
        
        x_score += np.select([df['rsi'] > 70, df['rsi'] > 50, df['rsi'] < 50], [2, 1, -1], default=0)
        result_df['X_Score'] = x_score

        # --- 計算 Y 軸 ---
        y_score = pd.Series(0, index=df.index)
        y_score += np.where(df['atr'] > df['atr_60d_avg'], 2, -2)
        y_score += np.select([df['bbw_percentile'] > 0.8, df['bbw_percentile'] < 0.2], [2, -2], default=0)
        y_score += np.where(df['vix_percentile'] >= 0.75, 1, -1)
        y_score += np.where(df['hv_percentile'] >= 0.75, 1, -1)
        result_df['Y_Score'] = y_score

        # --- 判斷象限 ---
        cond_quadrant = [
            (result_df['X_Score'] > 0) & (result_df['Y_Score'] > 0),
            (result_df['X_Score'] < 0) & (result_df['Y_Score'] > 0),
            (result_df['X_Score'] < 0) & (result_df['Y_Score'] < 0),
            (result_df['X_Score'] > 0) & (result_df['Y_Score'] < 0)
        ]
        result_df['Quadrant'] = np.select(cond_quadrant, [1, 2, 3, 4], default=0)

        return result_df

    def attach_descriptions(self, df: pd.DataFrame) -> pd.DataFrame:
        """將象限代號轉換為文字描述的擴充方法"""
        if 'Quadrant' not in df.columns:
            raise ValueError("請先執行 analyze_dataframe() 產生 Quadrant 欄位")
            
        result_df = df.copy()
        default_text = "位於原點或座標軸上(0)，代表象限不明，不予操作。"
        
        # 批量映射文字
        for key in ['title', 'features', 'targets', 'strategy']:
            result_df[f'quadrant_{key}'] = result_df['Quadrant'].map(
                lambda x: self.quadrant_info.get(x, {}).get(key, default_text)
            )
        return result_df

In [ ]:
#2026.3.17修改後
import pandas as pd
import numpy as np

class MarketQuadrantAnalyzer:
    def __init__(self):
        # 定義各象限的輸出文案
        self.quadrant_info = {
            1: {
                "title": "第一象限 (右上)：擴張 (+) + 高波動 (+) ——「狂熱末升段」",
                "features": "市場特徵： 景氣很好，價格不便宜，多空分歧大，洗盤頻繁。",
                "targets": "代表標的： 題材最熱、成交量暴增的強勢股。",
                "strategy": "策略： 動能策略。只要趨勢沒斷就追，嚴格執行移動止損。"
            },
            2: {
                "title": "第二象限 (左上)：收縮 (-) + 高波動 (+) ——「危機恐慌期」",
                "features": "市場特徵： 基本面轉壞，伴隨市場恐慌拋售。",
                "targets": "代表標的： 遭遇黑天鵝事件的大盤，或是產業崩跌期。",
                "strategy": "策略： 均值回歸或放空。利用超跌後的反彈獲利。"
            },
            3: {
                "title": "第三象限 (左下)：收縮 (-) + 低波動 (-) ——「蕭條築底期」",
                "features": "市場特徵： 市場極度冷清，乏人問津。",
                "targets": "代表標的： 傳統產業、高股息、或谷底盤整的週期股。",
                "strategy": "策略： 價值投資。尋找 P/B < 1 且現金流穩定的公司。"
            },
            4: {
                "title": "第四象限 (右下)：擴張 (+) + 低波動 (-) ——「金髮女孩」",
                "features": "市場特徵： 經濟穩步復甦，通膨適中，最舒服的送分題區間。",
                "targets": "代表標的： 進入穩定成長期的權值股、獲利翻正的科技股。",
                "strategy": "策略： 趨勢追蹤。量化模型最應優先捕捉的區間。"
            }
        }

    def analyze_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """執行向量化運算，計算 X, Y 分數與所屬象限"""
        result_df = df.copy()
        
        # 確保欄位名稱為小寫以對齊 Indicators 模組
        result_df.columns = [col.lower() for col in result_df.columns]
        
        # --- 計算 X 軸 (Expansion vs Contraction) ---
        x_score = pd.Series(0, index=result_df.index)
        
        # 價格與均線關係
        x_score += np.select([result_df['adj_price'] > result_df['ma200'], 
                              result_df['adj_price'] < result_df['ma200']], [2, -2], default=0)
        
        # 趨勢強度 ADX
        x_score += np.select([result_df['adx'] > 25, result_df['adx'] < 20], [1, -1], default=0)
        
        # 營收成長 YoY (優化：處理 NaN，加速成長 +2, 連續衰退 -2, 其餘 0)
        cond_yoy_up = (result_df['yoy_now'] > result_df['yoy_t1']) & (result_df['yoy_t1'] > result_df['yoy_t2'])
        cond_yoy_down = (result_df['yoy_now'] < result_df['yoy_t1']) & (result_df['yoy_t1'] < result_df['yoy_t2'])
        x_score += np.select([cond_yoy_up, cond_yoy_down], [2, -2], default=0)
        
        # 強弱指標 RSI
        x_score += np.select([result_df['rsi'] > 70, result_df['rsi'] > 50, result_df['rsi'] < 50], [2, 1, -1], default=0)
        
        result_df['x_score'] = x_score

        # --- 計算 Y 軸 (High Vol vs Low Vol) ---
        y_score = pd.Series(0, index=result_df.index)
        
        # 短期 ATR vs 長期平均
        y_score += np.where(result_df['atr'] > result_df['atr_60d_avg'], 2, -2)
        
        # 布林頻寬百分位
        y_score += np.select([result_df['bbw_percentile'] > 0.8, result_df['bbw_percentile'] < 0.2], [2, -2], default=0)
        
        # VIX 與 歷史波動率 (使用 0.75 作為高波動閾值)
        y_score += np.where(result_df['vix_percentile'] >= 0.75, 1, -1)
        y_score += np.where(result_df['hv_percentile'] >= 0.75, 1, -1)
        
        result_df['y_score'] = y_score

        # --- 判斷象限 ---
        cond_quadrant = [
            (result_df['x_score'] > 0) & (result_df['y_score'] > 0), # Q1
            (result_df['x_score'] < 0) & (result_df['y_score'] > 0), # Q2
            (result_df['x_score'] < 0) & (result_df['y_score'] < 0), # Q3
            (result_df['x_score'] > 0) & (result_df['y_score'] < 0)  # Q4
        ]
        result_df['quadrant'] = np.select(cond_quadrant, [1, 2, 3, 4], default=0)

        return result_df

    def attach_descriptions(self, df: pd.DataFrame) -> pd.DataFrame:
        """將象限代號轉換為文字描述的擴充方法"""
        # 統一檢查小寫欄位
        target_col = 'quadrant' if 'quadrant' in df.columns else 'Quadrant'
        if target_col not in df.columns:
            raise ValueError("請先執行 analyze_dataframe() 產生 quadrant 欄位")
            
        result_df = df.copy()
        default_text = "位於原點或座標軸上(0)，代表象限不明，不予操作。"
        
        # 批量映射文字
        for key in ['title', 'features', 'targets', 'strategy']:
            result_df[f'quadrant_{key}'] = result_df[target_col].map(
                lambda x: self.quadrant_info.get(x, {}).get(key, default_text)
            )
        return result_df